# 07 MidiKeyboardWidget Tutorial

Use a browser-connected MIDI keyboard as a note input source for sequencers and samplers.


## Connect a MIDI Device

Render the widget, choose a MIDI input port in the browser, then play notes. Velocity is preserved in `last_note_event` and broadcast to other widgets.


In [ ]:
from nbplay import MidiKeyboardWidget

midi_keyboard = MidiKeyboardWidget()
midi_keyboard

## Record into a Sequencer

`connect_sequencer()` enables the sequencer recording controls and lets MIDI note events fill steps while the sequencer is playing.


In [ ]:
from nbplay import SequencerWidget

record_seq = SequencerWidget(length=8, bpm=112.0)
midi_keyboard.connect_sequencer(record_seq)
print(record_seq.keyboard_connected)
record_seq

## Play a Sampler

Connect the MIDI widget to a sampler track in a session to route incoming notes through the same browser audio bus as the typing keyboard.

In [ ]:
from nbplay import SamplerWidget, Session

sampler_session = Session(bpm=112.0)
sampler = SamplerWidget(pad_count=12)
sampler.load_sample([0.0, 0.25, 0.5, 0.25, 0.0, -0.25, -0.5, -0.25], root_note=60, name='Tiny wave')
sampler_seq = SequencerWidget(length=8)
sampler_session.add_track('MIDI Sampler', sampler_seq, sampler)

midi_keyboard.session_id = sampler.session_id
midi_keyboard.connect_sampler(sampler, zone='all')
print(midi_keyboard.sampler_routing)
sampler

## Split Samplers by Register

Multiple samplers can share one MIDI keyboard. Use upper/lower zones to create a simple octave-style split.

In [ ]:
split_session = Session(bpm=96.0)
lower_sampler = SamplerWidget(sample_name='Lower sampler', root_note=48, pad_count=8)
upper_sampler = SamplerWidget(sample_name='Upper sampler', root_note=72, pad_count=8)
split_session.add_track('Lower', SequencerWidget(length=8), lower_sampler)
split_session.add_track('Upper', SequencerWidget(length=8), upper_sampler)

midi_split = MidiKeyboardWidget(session_id=split_session.mixer.session_id)
midi_split.connect_sampler(lower_sampler, zone='lower')
midi_split.connect_sampler(upper_sampler, zone='upper')
print(midi_split.sampler_routing)
midi_split